# Fetching data at warp speed

In [2]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD"]
MONTHS = [
    "202101", "202102", "202103", "202104", "202105", "202106",
    "202107", "202108", "202109", "202110", "202111", "202112",
    "202201", "202202", "202203", "202204", "202205", "202206",
    "202207", "202208", "202209", "202210", "202211", "202212"]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Parallel Download of 48 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2021-03-01T08:37:05.914000
INFO:DUKASCRIPT:current timestamp :2021-01-04T08:04:26.409000
INFO:DUKASCRIPT:current timestamp :2021-04-01T10:08:04.013000
INFO:DUKASCRIPT:current timestamp :2021-02-01T08:05:14.449000
INFO:DUKASCRIPT:current timestamp :2021-03-01T15:06:43.885000
INFO:DUKASCRIPT:current timestamp :2021-01-04T15:38:06.123000
INFO:DUKASCRIPT:current timestamp :2021-02-01T13:22:55.882000
INFO:DUKASCRIPT:current timestamp :2021-04-01T23:10:01.873000
INFO:DUKASCRIPT:current timestamp :2021-03-02T01:06:53.580000
INFO:DUKASCRIPT:current timestamp :2021-01-05T00:17:30.579000
INFO:DUKASCRIPT:current timestamp :2021-02-01T17:56:22.980000
INFO:DUKASCRIPT:current timestamp :2021-04-05T03:37:49.163000
INFO:DUKASCRIPT:current timestamp :2021-03-02T08:19:24.886000
INFO:DUKASCRIPT:current timestamp :2021-01-05T08:32:00.802000
INFO:DUKASCRIPT:current timestamp :2021-04-06T01:01:38.189000
INFO:DUKASCRIPT:current timestamp :2021-02-02T04:36:08.107000
INFO:DUK

✅ Success: AUD/USD 202101 -> audusd_dukascopy_bid_202101.parquet (1,812,242 rows) & audusd_dukascopy_ask_202101.parquet (1,812,242 rows)
✅ Success: AUD/USD 202102 -> audusd_dukascopy_bid_202102.parquet (1,474,801 rows) & audusd_dukascopy_ask_202102.parquet (1,474,801 rows)
✅ Success: AUD/USD 202103 -> audusd_dukascopy_bid_202103.parquet (1,881,111 rows) & audusd_dukascopy_ask_202103.parquet (1,881,111 rows)
✅ Success: AUD/USD 202104 -> audusd_dukascopy_bid_202104.parquet (1,158,076 rows) & audusd_dukascopy_ask_202104.parquet (1,158,076 rows)
✅ Success: AUD/USD 202105 -> audusd_dukascopy_bid_202105.parquet (1,253,583 rows) & audusd_dukascopy_ask_202105.parquet (1,253,583 rows)
✅ Success: AUD/USD 202106 -> audusd_dukascopy_bid_202106.parquet (1,193,980 rows) & audusd_dukascopy_ask_202106.parquet (1,193,980 rows)
✅ Success: AUD/USD 202107 -> audusd_dukascopy_bid_202107.parquet (1,446,699 rows) & audusd_dukascopy_ask_202107.parquet (1,446,699 rows)
✅ Success: AUD/USD 202108 -> audusd_dukas